# 03 合并最终输出表

**目标**: 将原始宽表 + 社区归属 + 模型打标结果合并为一张完整的 CSV，供业务方直接使用。

**输出列顺序**:
1. `device_id` — 设备号
2. `community_id` — Leiden 社区归属（紧跟 device_id）
3. 原始宽表全部字段
4. 派生特征（refund_rate / comp_amount_rate / is_short_refund_strong / is_short_refund_weak / is_machine_refund / is_night_heavy / is_multi_account / is_multi_pay_tool / is_multi_passenger）
5. 六模型异常分数 + 打标结果
6. 投票数 + 规则命中数
7. `pseudo_label` — 伪标签
8. `risk_level` — 最终风险分层

In [ ]:
import os, time
import pandas as pd
from tqdm import tqdm

# [TUNABLE] 修改范围: 数据/输出目录
BASE   = os.environ.get("LEIDEN_BASE", r"d:/Qunar_work/workbuddy_data/leiden")
DATA   = os.path.join(BASE, "data")
OUTPUT = os.path.join(DATA, "model_output")

# [TUNABLE] 输入文件名
INPUT_CSV    = os.path.join(DATA, "flight_feature_detail_8.19-90days.csv")
COMM_CSV     = os.path.join(OUTPUT, "device_community.csv")
RISK_CSV     = os.path.join(OUTPUT, "device_risk_score.csv")
# [TUNABLE] 输出文件名
MERGED_CSV   = os.path.join(OUTPUT, "final_merged_output.csv")

print("配置:")
print(f"  原始宽表: {INPUT_CSV}")
print(f"  社区归属: {COMM_CSV}")
print(f"  风险评分: {RISK_CSV}")
print(f"  最终输出: {MERGED_CSV}")

## 1. 加载三张表

In [ ]:
print("[1/4] 加载数据")
t0 = time.time()

print("  加载原始宽表...")
# [TUNABLE] dtype=str: prevent device_id scientific notation
# Impact: if changed to auto-detect, device_id may become 8.65E+14
df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8", dtype=str)
print(f"    {len(df_raw)} 行, {len(df_raw.columns)} 列, 耗时 {time.time()-t0:.1f}s")
# --- 数据清洗: 去除脏值 (null/nan/None/空字符串) ---
# [TUNABLE] 修改范围: 可增删需要清洗的脏值关键词
# 影响内容: 清洗不干净 -> 脏值进入合并输出; 清洗过度 -> 丢失真实关联
DIRTY_VALUES = {"null", "nan", "none", "n/a", "na", "NULL", "NaN", "None", "N/A", "", " "}
for col in df_raw.columns:
    if df_raw[col].dtype == object:
        df_raw[col] = df_raw[col].astype(str).str.strip()
        df_raw[col] = df_raw[col].replace({k: None for k in DIRTY_VALUES})
        df_raw[col] = df_raw[col].where(df_raw[col].notna() & (df_raw[col].str.lower() != 'none') & (df_raw[col].str.lower() != 'nan'), None)
print(f"  数据清洗完成, {len(df_raw)} 行")

# 去重: 同一 device_id 保留第一条（原始宽表可能因JOIN产生重复行）
dup_cnt = df_raw["device_id"].duplicated().sum()
if dup_cnt > 0:
    print(f"    [警告] 原始宽表存在 {dup_cnt} 个重复 device_id，保留第一条")
    df_raw = df_raw.drop_duplicates(subset=["device_id"], keep="first")
    print(f"    去重 {dup_cnt} 行 -> {len(df_raw)} 行")

t1 = time.time()
print("  加载社区归属...")
df_comm = pd.read_csv(COMM_CSV, encoding="utf-8-sig")
# 只取设备节点
df_comm = df_comm[df_comm["node_type"] == "device"].copy()
df_comm = df_comm[["node", "community_id"]].rename(columns={"node": "device_id"})
print(f"    {len(df_comm)} 设备节点, 耗时 {time.time()-t1:.1f}s")

t1 = time.time()
print("  加载风险评分...")
df_risk = pd.read_csv(RISK_CSV, encoding="utf-8-sig")
print(f"    {len(df_risk)} 设备, {len(df_risk.columns)} 列, 耗时 {time.time()-t1:.1f}s")
# --- 数据清洗: 去除脏值 (null/nan/None/空字符串) ---
# [TUNABLE] 修改范围: 可增删需要清洗的脏值关键词
# 影响内容: 清洗不干净 -> 脏值进入合并输出; 清洗过度 -> 丢失真实关联
DIRTY_VALUES = {"null", "nan", "none", "n/a", "na", "NULL", "NaN", "None", "N/A", "", " "}
for col in df_risk.columns:
    if df_risk[col].dtype == object:
        df_risk[col] = df_risk[col].astype(str).str.strip()
        df_risk[col] = df_risk[col].replace({k: None for k in DIRTY_VALUES})
        df_risk[col] = df_risk[col].where(df_risk[col].notna() & (df_risk[col].str.lower() != 'none') & (df_risk[col].str.lower() != 'nan'), None)
print(f"  数据清洗完成, {len(df_risk)} 行")

# 去重: 同一 device_id 保留第一条
risk_dup = df_risk["device_id"].duplicated().sum()
if risk_dup > 0:
    print(f"    [警告] 风险评分表存在 {risk_dup} 个重复 device_id，保留第一条")
    df_risk = df_risk.drop_duplicates(subset=["device_id"], keep="first")
    print(f"    去重 {risk_dup} 行 -> {len(df_risk)} 行")

## 2. 合并

合并逻辑:
- 左连接原始宽表 <- 社区归属（device_id 为 key）
- 左连接原始宽表 <- 风险评分（device_id 为 key，只取新增列）
- 未匹配到社区/风险的设备，community_id 填 -1，模型列填默认值

In [ ]:
print("[2/4] 合并")
t0 = time.time()

# 合并社区归属 (left join)
df = df_raw.merge(df_comm, on="device_id", how="left")
print(f"  社区匹配: {df['community_id'].notna().sum()} / {len(df)}")

# 原始表中已有的列不重复引入
# [TUNABLE] 需要从风险评分表追加的列（删减会影响最终表包含哪些模型结果）
risk_new_cols = [
    "refund_rate", "comp_amount_rate", "is_short_refund_strong", "is_short_refund_weak",
    "is_machine_refund", "is_night_heavy",
    "is_multi_account", "is_multi_pay_tool", "is_multi_passenger",
    "iforest_score", "iforest_anomaly",
    "ocsvm_score", "ocsvm_anomaly",
    "lof_score", "lof_anomaly",
    "xgb_prob", "xgb_pred",
    "lgb_prob", "lgb_pred",
    "rf_prob", "rf_pred",
    "vote_anomaly_cnt", "vote_total", "rule_hit_cnt",
    "pseudo_label", "risk_level",
]
# 确保这些列在风险评分表中存在
risk_new_cols = [c for c in risk_new_cols if c in df_risk.columns]
risk_subset = df_risk[["device_id"] + risk_new_cols]

# 合并风险评分 (left join)
df = df.merge(risk_subset, on="device_id", how="left")
print(f"  风险匹配: {df['risk_level'].notna().sum()} / {len(df)}")

# 填充未匹配的默认值
df["community_id"] = df["community_id"].fillna(-1).astype(int)
for col in ["iforest_anomaly", "ocsvm_anomaly", "lof_anomaly",
            "xgb_pred", "lgb_pred", "rf_pred", "is_short_refund_strong", "is_short_refund_weak",
            "is_machine_refund", "is_night_heavy",
            "is_multi_account", "is_multi_pay_tool", "is_multi_passenger"]:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)
df["pseudo_label"] = df["pseudo_label"].fillna(-1).astype(int)
df["risk_level"] = df["risk_level"].fillna("未评估")

print(f"  合并完成: {len(df)} 行, {len(df.columns)} 列, 耗时 {time.time()-t0:.1f}s")

## 3. 调整列顺序

`device_id` -> `community_id` -> 原始全部列 -> 派生+模型+投票+标签+风险分层

In [ ]:
print("[3/4] 调整列顺序")
# device_id 和 community_id 放最前
front = ["device_id", "community_id"]
# 原始列（去掉 device_id，它在 front 里了）
orig_cols = [c for c in df_raw.columns if c != "device_id"]
# 新增列（去掉 community_id，已在 front 里了）
new_cols = [c for c in risk_new_cols if c not in ["community_id"]]
final_cols = front + orig_cols + new_cols
# 安全检查：确保所有列都存在
final_cols = [c for c in final_cols if c in df.columns]
df = df[final_cols]
print(f"  最终列数: {len(df.columns)}")
print(f"  列顺序: {final_cols[:5]} ... {final_cols[-5:]}")

## 4. 输出

统一 UTF-8-SIG 编码（Excel 可直接打开）。

In [ ]:
print("[4/4] 输出")
t0 = time.time()
df.to_csv(MERGED_CSV, index=False, encoding="utf-8-sig")
fsize = os.path.getsize(MERGED_CSV) / 1024 / 1024
print(f"  输出: {MERGED_CSV}")
print(f"  大小: {fsize:.1f} MB, 耗时 {time.time()-t0:.1f}s")
print(f"  最终表: {len(df)} 行 x {len(df.columns)} 列")

# 快速验证
print("\n=== 验证 ===")
print(f"  community_id 非空: {df[df['community_id'] != -1].shape[0]} ({df[df['community_id'] != -1].shape[0]/len(df)*100:.1f}%)")
print(f"  risk_level 分布:")
print(df["risk_level"].value_counts().to_string())
print(f"\n  前3行 device_id / community_id / risk_level:")
print(df[["device_id", "community_id", "risk_level"]].head(3).to_string())